<a href="https://colab.research.google.com/github/trash-taste/krisha-kz-mlops/blob/main/almaty_2024.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🏙️ Almaty Real Estate AVM (Automated Valuation Model)

![Python](https://img.shields.io/badge/python-3.9+-blue.svg)
![CatBoost](https://img.shields.io/badge/CatBoost-1.2-yellow.svg)
![Scikit-Learn](https://img.shields.io/badge/scikit--learn-1.3+-orange.svg)
![Status](https://img.shields.io/badge/Status-Production_Ready-success.svg)

## 📌 Business Problem
Оценка недвижимости в развивающихся рынках (Алматы) осложнена высоким уровнем шума в данных: фейковые объявления, экстремальные выбросы и отсутствие точных гео-координат.
**Цель проекта:** Создать интерпретируемую ML-систему (AVM) для автоматической оценки рыночной стоимости квартир, которую можно интегрировать в backend (API) для использования риелторами и клиентами.

## ⚙️ Architecture & Pipeline Design
Проект спроектирован по стандартам **Production ML**, строго разделяя трансформации без состояния и трансформации, зависящие от данных, для предотвращения Data Leakage (утечки данных).

1. **Stateless Preprocessing:** Санитарная фильтрация (удаление опечаток парсинга через IQR), приведение типов, создание базовых эвристик (proxy features: `near_metro`, `location_abay`).
2. **Strict Data Split:** Разделение данных ДО применения любых stateful-трансформаций.
3. **Stateful Feature Engineering (Custom Transformers):**
   - `SmoothedTargetEncoder`: Байесовское сглаживание таргета по микро-локациям для борьбы с переобучением на редких районах.
   - `PriceTierBinner`: Дискретизация закодированных локаций на 5 ценовых сегментов (Market Tiers).
4. **Outlier Isolation (Train Only):** Использование масштабированного `IsolationForest` для удаления многомерных аномалий исключительно из обучающей выборки (чтобы модель умела работать с "грязными" запросами в проде).
5. **Modeling:** Трансформация таргета (`price_per_sqm` вместо `total_price`) для снижения дисперсии. Основная модель — `CatBoostRegressor`.

## 📊 Model Benchmarking (5-Fold CV)
Перед выбором финальной модели был проведен бейзлайн-тест на очищенных данных:

| Model | CV MAPE (Score) | Std Dev |
| :--- | :--- | :--- |
| Linear Regression | 16.30% | ± 0.56% |
| Random Forest | 14.24% | ± 0.40% |
| **CatBoost (Champion)** | **13.70%** | **Holdout Test** |

*Бизнес-результат:* Модель ошибается в среднем на **13.7%** (MAE: ~4.3 млн ₸), что укладывается в стандарты "вилки для торга" живого риелтора (10-15%).

## 🧠 Explainability (SHAP Insights)
Модель не является "черным ящиком". Ключевые инсайты рынка Алматы (см. `notebooks/02_model_experiments.ipynb`):
* **Площадь:** Главный драйвер цены, но имеет нелинейную зависимость (скидка за огромные площади).
* **Location Premium:** Фича `location_abay` (Выше Абая) статистически доказала наличие мощной ценовой премии за экологию и престиж, обходя формальное деление на административные районы.
* **Метро:** Влияние `near_metro` оказалось минимальным, что подтверждает автомобилецентричность рынка Алматы.

## 📦 Deployment & Inference
Модель и все стейтфул-трансформеры упакованы в единый артефакт `avm_artifacts.pkl` через `joblib`.
Пример использования (см. `src/inference.py`):
```python
from src.inference import RealEstateAVM

avm = RealEstateAVM(artifact_path='models/avm_artifacts.pkl')
prediction = avm.predict({
    "plosh_ob": 55.0,
    "gorod": "Алматы",
    "rayon": "Бостандыкский",
    "ulica": "проспект Абая"
})
print(f"Оценка: {prediction['total_price']} KZT")

In [26]:
# ==============================================================================
# АВТОМАТИЧЕСКАЯ СБОРКА GITHUB-РЕПОЗИТОРИЯ: ALMATY AVM
# ==============================================================================
import os

# 1. ПРОВЕРКА НАЛИЧИЯ ДАННЫХ
if not os.path.exists('Data_2024_Almaty_kv.csv'):
    raise FileNotFoundError("Сначала загрузите 'Data_2024_Almaty_kv.csv' в корень Colab!")

# 2. СОЗДАНИЕ СТРУКТУРЫ
!mkdir -p almaty-avm/src
!mkdir -p almaty-avm/models
!mkdir -p almaty-avm/data_sample
!mv Data_2024_Almaty_kv.csv almaty-avm/data_sample/

# 3. REQUIREMENTS
with open('almaty-avm/requirements.txt', 'w') as f:
    f.write("pandas>=2.0.0\nnumpy>=1.24.0\nscikit-learn>=1.3.0\ncatboost>=1.2\njoblib>=1.3.0\n")

# 4. ТРАНСФОРМЕРЫ (src/transformers.py)
with open('almaty-avm/src/transformers.py', 'w') as f:
    f.write("""import pandas as pd
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import KBinsDiscretizer

class SmoothedTargetEncoder(BaseEstimator, TransformerMixin):
    def __init__(self, cat_col, alpha=10):
        self.cat_col = cat_col
        self.alpha = alpha
        self.mapping_ = {}
        self.global_target_ = None

    def fit(self, X, y):
        self.global_target_ = y.median()
        stats = y.groupby(X[self.cat_col]).agg(['median', 'count'])
        weight = stats['count'] / (stats['count'] + self.alpha)
        smoothed = weight * stats['median'] + (1 - weight) * self.global_target_
        self.mapping_ = smoothed.to_dict()
        return self

    def transform(self, X):
        X_out = X.copy()
        X_out[f'{self.cat_col}_encoded'] = X_out[self.cat_col].map(self.mapping_).fillna(self.global_target_)
        return X_out

class PriceTierBinner(BaseEstimator, TransformerMixin):
    def __init__(self, col, n_bins=5):
        self.col = col
        self.binner = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='kmeans')

    def fit(self, X, y=None):
        self.binner.fit(X[[self.col]])
        return self

    def transform(self, X):
        X_out = X.copy()
        X_out[f'{self.col}_tier'] = self.binner.transform(X_out[[self.col]]).astype(int)
        return X_out
""")

# 5. ОБУЧЕНИЕ (src/train.py)
with open('almaty-avm/src/train.py', 'w') as f:
    f.write("""import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_percentage_error, mean_absolute_error, r2_score
from catboost import CatBoostRegressor
import warnings
warnings.filterwarnings('ignore')

from transformers import SmoothedTargetEncoder, PriceTierBinner

print("🏭 ZONE 1: Stateless Data Cleaning...")
df = pd.read_csv('../data_sample/Data_2024_Almaty_kv.csv')
df = df.dropna(subset=['summa', 'plosh_ob']).copy()
df['summa'] = pd.to_numeric(df['summa'], errors='coerce')
df['plosh_ob'] = pd.to_numeric(df['plosh_ob'], errors='coerce')
df = df.dropna(subset=['summa', 'plosh_ob'])

df['price_per_sqm'] = df['summa'] / df['plosh_ob']
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=['price_per_sqm'])

Q1 = df['price_per_sqm'].quantile(0.25)
Q3 = df['price_per_sqm'].quantile(0.75)
MAX_SQM_PRICE = Q3 + 1.5 * (Q3 - Q1)

df = df[(df['price_per_sqm'] >= 300_000) & (df['price_per_sqm'] <= MAX_SQM_PRICE) &
        (df['summa'] >= 5_000_000) & (df['plosh_ob'] >= 15) & (df['plosh_ob'] <= df['plosh_ob'].quantile(0.99))].copy()

metro_streets = ['Абая', 'Назарбаев', 'Фурманова', 'Гагарина', 'Розыбакиева', 'Сейфуллина']
df['near_metro'] = df['ulica'].apply(lambda x: 1 if any(m.lower() in str(x).lower() for m in metro_streets) else 0)

upper_dist = ['Бостандыкский', 'Медеуский', 'Наурызбайский']
upper_micr = ['Таугуль', 'Мамыр', 'Мирас', 'Самал', 'Коктем', 'Орбита', 'Казахфильм', 'Ремизовка']
df['location_abay'] = df['rayon'].apply(lambda r: 1 if any(d in str(r) for d in upper_dist) or any(m in str(r) for m in upper_micr) else 0)
df['micro_loc'] = df['gorod'].astype(str) + " | " + df['rayon'].astype(str)

print("✂️ ZONE 2: Strict Train/Test Split...")
X = df[['plosh_ob', 'gorod', 'rayon', 'micro_loc', 'location_abay', 'near_metro']]
y_sqm = df['price_per_sqm']
y_tot = df['summa']
X_train, X_test, y_train_sqm, y_test_sqm, y_train_tot, y_test_tot = train_test_split(X, y_sqm, y_tot, test_size=0.2, random_state=42)

print("🧠 ZONE 3: Stateful FE...")
encoder = SmoothedTargetEncoder(cat_col='micro_loc', alpha=15)
X_train_tf = encoder.fit_transform(X_train, y_train_sqm)
X_test_tf = encoder.transform(X_test)

binner = PriceTierBinner(col='micro_loc_encoded', n_bins=5)
X_train_tf = binner.fit_transform(X_train_tf)
X_test_tf = binner.transform(X_test_tf)

print("🤖 ZONE 4: Isolation Forest (Train Only)...")
isf_features = ['plosh_ob', 'micro_loc_encoded', 'location_abay']
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_tf[isf_features])
isf = IsolationForest(contamination=0.03, random_state=42)
mask = isf.fit_predict(X_train_scaled) == 1

X_train_clean = X_train_tf[mask].copy()
y_train_clean = y_train_sqm[mask].copy()

print("🚀 ZONE 5: Training CatBoost...")
cb_features = ['plosh_ob', 'micro_loc_encoded', 'location_abay', 'near_metro', 'micro_loc_encoded_tier', 'gorod', 'rayon']
model = CatBoostRegressor(iterations=1500, learning_rate=0.05, depth=6, cat_features=['gorod', 'rayon'], random_seed=42, verbose=0)
model.fit(X_train_clean[cb_features], y_train_clean, eval_set=(X_test_tf[cb_features], y_test_sqm), early_stopping_rounds=50)

print("📊 ZONE 6: Business Evaluation...")
pred_sqm = model.predict(X_test_tf[cb_features])
pred_tot = pred_sqm * X_test_tf['plosh_ob'].values
mape = mean_absolute_percentage_error(y_test_tot, pred_tot) * 100
print(f"🏆 MAPE на Test Set: {mape:.1f}%")

print("💾 ZONE 7: Saving Artifacts...")
os.makedirs('../models', exist_ok=True)
joblib.dump({"target_encoder": encoder, "tier_binner": binner, "model": model}, '../models/avm_artifacts.pkl')
print("✅ Успешно!")
""")

# 6. ИНФЕРЕНС (src/inference.py)
with open('almaty-avm/src/inference.py', 'w') as f:
    f.write("""import pandas as pd
import joblib

class RealEstateAVM:
    def __init__(self, artifact_path='../models/avm_artifacts.pkl'):
        artifacts = joblib.load(artifact_path)
        self.encoder = artifacts['target_encoder']
        self.binner = artifacts['tier_binner']
        self.model = artifacts['model']
        self.metro_streets = ['Абая', 'Назарбаев', 'Фурманова', 'Гагарина', 'Розыбакиева', 'Сейфуллина']
        self.upper_dist = ['Бостандыкский', 'Медеуский', 'Наурызбайский']
        self.upper_micr = ['Таугуль', 'Мамыр', 'Мирас', 'Самал', 'Коктем', 'Орбита', 'Казахфильм', 'Ремизовка']

    def predict(self, raw_data: dict) -> dict:
        df = pd.DataFrame([raw_data])
        df['near_metro'] = df['ulica'].apply(lambda x: 1 if any(m.lower() in str(x).lower() for m in self.metro_streets) else 0)
        df['location_abay'] = df['rayon'].apply(lambda r: 1 if any(d in str(r) for d in self.upper_dist) or any(m in str(r) for m in self.upper_micr) else 0)
        df['micro_loc'] = df['gorod'].astype(str) + " | " + df['rayon'].astype(str)

        df = self.encoder.transform(df)
        df = self.binner.transform(df)

        features = ['plosh_ob', 'micro_loc_encoded', 'location_abay', 'near_metro', 'micro_loc_encoded_tier', 'gorod', 'rayon']
        pred_sqm = self.model.predict(df[features])[0]
        total_price = pred_sqm * raw_data['plosh_ob']
        return {"price_sqm": int(pred_sqm), "total_price": int(total_price)}

if __name__ == "__main__":
    avm = RealEstateAVM()
    test_flat = {"plosh_ob": 65.0, "gorod": "Алматы", "rayon": "Бостандыкский", "ulica": "Гагарина"}
    result = avm.predict(test_flat)
    print(f"💰 ИНФЕРЕНС УСПЕШЕН! Прогноз стоимости: {result['total_price']:,} KZT")
""")

# 7. ЗАПУСК И УПАКОВКА (MAGIC COMMANDS)
print("\n" + "="*50)
print("📥 Установка библиотек...")
!pip install -q -r almaty-avm/requirements.txt

print("\n🚀 Обучение модели...")
%cd almaty-avm/src
!python train.py

print("\n✅ Тестирование инференса...")
!python inference.py

print("\n📦 Упаковка в ZIP...")
%cd /content
!zip -r -q almaty-avm.zip almaty-avm
print("\n🎉 ГОТОВО! Скачайте файл 'almaty-avm.zip' из левого меню файлов.")


📥 Установка библиотек...

🚀 Обучение модели...
/content/almaty-avm/src
🏭 ZONE 1: Stateless Data Cleaning...
✂️ ZONE 2: Strict Train/Test Split...
🧠 ZONE 3: Stateful FE...
🤖 ZONE 4: Isolation Forest (Train Only)...
🚀 ZONE 5: Training CatBoost...
📊 ZONE 6: Business Evaluation...
🏆 MAPE на Test Set: 13.7%
💾 ZONE 7: Saving Artifacts...
✅ Успешно!

✅ Тестирование инференса...
💰 ИНФЕРЕНС УСПЕШЕН! Прогноз стоимости: 44,720,827 KZT

📦 Упаковка в ZIP...
/content

🎉 ГОТОВО! Скачайте файл 'almaty-avm.zip' из левого меню файлов.
